In [1]:
!pip install pandas faker reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 86.3 MB/s eta 0:00:00


In [4]:
import pandas as pd
from reportlab.pdfgen import canvas
from faker import Faker
import os

fake = Faker()

def generate_test_data(match=True):

    po_no = "PO-12345"
    item = "Industrial Sensor"
    qty = 10
    price = 250

    df_po = pd.DataFrame([{
        'po_number': po_no,
        'item': item,
        'quantity': qty,
        'unit_price': price
    }])
    df_po.to_csv('purchase_orders.csv', index=False)

    invoice_price = price if match else price + 50

    c = canvas.Canvas("invoice_test.pdf")
    c.drawString(100, 750, f"INVOICE #INV-001")
    c.drawString(100, 730, f"PO Reference: {po_no}")
    c.drawString(100, 710, f"Item: {item}")
    c.drawString(100, 690, f"Quantity: {qty}")
    c.drawString(100, 670, f"Unit Price: ${invoice_price}")
    c.drawString(100, 650, f"Total Due: ${qty * invoice_price}")
    c.save()

    print(f"Files created! Mismatch triggered: {not match}")


generate_test_data(match=False)

Files created! Mismatch triggered: True


In [6]:
!pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 130.0 MB/s eta 0:00:00


In [7]:
import pdfplumber
import pandas as pd

def start_reconciliation(pdf_path, csv_path):

    po_db = pd.read_csv(csv_path)


    extracted_data = {}
    with pdfplumber.open(pdf_path) as pdf:
        page = pdf.pages[0]
        text = page.extract_text()


        for line in text.split('\n'):
            if "PO Reference:" in line:
                extracted_data['po_ref'] = line.split(': ')[1].strip()
            if "Unit Price: $" in line:
                extracted_data['price'] = float(line.split('$')[1].strip())


    matching_po = po_db[po_db['po_number'] == extracted_data['po_ref']]

    if matching_po.empty:
        print(f"❌ ERROR: PO {extracted_data['po_ref']} not found in database!")
    else:
        actual_price = matching_po.iloc[0]['unit_price']
        billed_price = extracted_data['price']

        if actual_price == billed_price:
            print(f"✅ SUCCESS: Invoice matches PO {extracted_data['po_ref']}. Ready for payment.")
        else:
            diff = billed_price - actual_price
            print(f"⚠️ DISCREPANCY FOUND!")
            print(f"Details: Billed ${billed_price}, but PO price is ${actual_price}.")
            print(f"Action: Flagging for ${diff} overcharge.")


start_reconciliation("invoice_test.pdf", "purchase_orders.csv")

⚠️ DISCREPANCY FOUND!
Details: Billed $300.0, but PO price is $250.
Action: Flagging for $50.0 overcharge.


In [8]:
import pandas as pd
from reportlab.pdfgen import canvas
import os

def generate_multi_item_data():

    items = [
        {'id': 'LIT-01', 'name': 'MacBook Pro', 'qty': 5, 'price': 2000},
        {'id': 'LIT-02', 'name': 'Magic Mouse', 'qty': 10, 'price': 80},
        {'id': 'LIT-03', 'name': 'USB-C Hub', 'qty': 15, 'price': 50}
    ]
    df_po = pd.DataFrame(items)
    df_po.to_csv('multi_purchase_orders.csv', index=False)


    c = canvas.Canvas("multi_item_invoice.pdf")
    c.setFont("Helvetica-Bold", 14)
    c.drawString(100, 800, "INVOICE #INV-MULTI-2026")
    c.setFont("Helvetica", 10)
    c.drawString(100, 780, "PO Reference: PO-MULTI-999")

    y = 750
    c.drawString(100, y, "Item ID | Name | Qty | Price")
    y -= 20


    invoice_items = [
        "LIT-01 | MacBook Pro | 6 | $2000", # Wrong Qty
        "LIT-02 | Magic Mouse | 10 | $80",   # Correct
        "LIT-03 | USB-C Hub | 15 | $60"      # Wrong Price
    ]

    for line in invoice_items:
        c.drawString(100, y, line)
        y -= 20

    c.save()
    print("Multi-item PO and Mismatched Invoice created!")

generate_multi_item_data()

Multi-item PO and Mismatched Invoice created!


In [9]:
import pdfplumber
import pandas as pd

def reconcile_multi_items(pdf_path, csv_path):
    po_db = pd.read_csv(csv_path)
    report = []

    with pdfplumber.open(pdf_path) as pdf:
        text = pdf.pages[0].extract_text()
        lines = text.split('\n')


        for line in lines:
            if "LIT-" in line:
                parts = [p.strip() for p in line.split('|')]
                item_id = parts[0]
                qty = int(parts[2])
                price = float(parts[3].replace('$', ''))


                match = po_db[po_db['id'] == item_id].iloc[0]

                status = "✅ Match"
                notes = ""


                if qty != match['qty'] or price != match['price']:
                    status = "⚠️ Discrepancy"
                    notes = f"Expected {match['qty']} @ ${match['price']}"

                report.append({
                    'Item': item_id,
                    'Billed Qty': qty,
                    'Billed Price': price,
                    'Status': status,
                    'Notes': notes
                })


    report_df = pd.DataFrame(report)
    print("\n--- AGENT RECONCILIATION AUDIT ---")
    print(report_df.to_string(index=False))

reconcile_multi_items("multi_item_invoice.pdf", "multi_purchase_orders.csv")


--- AGENT RECONCILIATION AUDIT ---
  Item  Billed Qty  Billed Price         Status              Notes
LIT-01           6        2000.0 ⚠️ Discrepancy Expected 5 @ $2000
LIT-02          10          80.0        ✅ Match                   
LIT-03          15          60.0 ⚠️ Discrepancy  Expected 15 @ $50
